[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-3/time-travel.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239536-lesson-5-time-travel)

# 时间旅行（Time Travel）

## 回顾

我们讨论了人机协同（human-in-the-loop）的动机：

(1) `Approval`（审批）—— 我们可以中断 Agent，将状态展示给用户，并允许用户批准某个操作

(2) `Debugging`（调试）—— 我们可以回退图的执行以重现或避免问题

(3) `Editing`（编辑）—— 你可以修改状态

我们展示了断点如何在特定节点停止图，或允许图动态地自我中断。

然后我们展示了如何进行人工审批继续执行，或通过人工反馈直接编辑图状态。

## 目标

现在，让我们展示 LangGraph 如何通过查看、重放甚至从过去状态分叉来[支持调试](https://docs.langchain.com/oss/python/langgraph/use-time-travel)。

我们称之为`时间旅行`（time travel）。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph langchain_deepseek langgraph_sdk langgraph-prebuilt

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

让我们构建我们的 Agent。

In [ ]:
from langchain_deepseek import ChatDeepSeek

def multiply(a: int, b: int) -> int:
    """计算 a 和 b 的乘积。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a * b

# 这将作为一个工具
def add(a: int, b: int) -> int:
    """计算 a 和 b 的和。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a + b

def divide(a: int, b: int) -> float:
    """计算 a 除以 b 的商。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a / b

tools = [add, multiply, divide]
llm = ChatDeepSeek(model="deepseek-chat")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from IPython.display import Image, display

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# 系统消息
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# 节点
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# 图
builder = StateGraph(MessagesState)

# 定义节点：这些节点执行实际工作
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# 定义边：这些边决定控制流
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # 如果来自 assistant 的最新消息（结果）是工具调用 -> tools_condition 路由到 tools
    # 如果来自 assistant 的最新消息（结果）不是工具调用 -> tools_condition 路由到 END
    tools_condition,
)
builder.add_edge("tools", "assistant")

memory = MemorySaver()
graph = builder.compile(checkpointer=MemorySaver())

# 展示
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

让我们像之前一样运行它。

In [ ]:
# 输入
initial_input = {"messages": HumanMessage(content="Multiply 2 and 3")}

# 线程
thread = {"configurable": {"thread_id": "1"}}

# 运行图直到第一次中断
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

## 浏览历史

我们可以使用 `get_state` 来查看给定 `thread_id` 下图的**当前**状态！

In [ ]:
graph.get_state({'configurable': {'thread_id': '1'}})

我们还可以浏览 Agent 的状态历史。

`get_state_history` 使我们能够获取所有先前步骤的状态。


In [ ]:
all_states = [s for s in graph.get_state_history(thread)]

In [ ]:
len(all_states)

第一个元素就是当前状态，与我们通过 `get_state` 获取的一致。

In [ ]:
all_states[-2]

以上所有内容我们可以在这里可视化：

![fig1.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbb038211b544898570be3_time-travel1.png)

## 重放（Replaying）

我们可以从任何先前的步骤重新运行我们的 Agent。

![fig2.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbb038a0bd34b541c78fb8_time-travel2.png)

让我们回顾一下接收了人工输入的那个步骤！

In [ ]:
to_replay = all_states[-2]

In [ ]:
to_replay

查看状态。

In [ ]:
to_replay.values

我们可以看到下一个要调用的节点。

In [ ]:
to_replay.next

我们还获取了 config，它告诉我们 `checkpoint_id` 和 `thread_id`。

In [ ]:
to_replay.config

要从这里重放，我们只需将 config 传回给 Agent！

图知道此检查点已经执行过。

它只是从此检查点重放！

In [ ]:
for event in graph.stream(None, to_replay.config, stream_mode="values"):
    event['messages'][-1].pretty_print()

现在，我们可以看到 Agent 重新运行后的当前状态。

## 分叉（Forking）

如果我们想从同一个步骤运行，但使用不同的输入呢？

这就是分叉。

![fig3.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbb038f89f2d847ee5c336_time-travel3.png)

In [ ]:
to_fork = all_states[-2]
to_fork.values["messages"]

再次，我们有了 config。

In [ ]:
to_fork.config

让我们修改此检查点的状态。

我们只需在提供 `checkpoint_id` 的情况下运行 `update_state`。

请记住我们的 `messages` reducer 是如何工作的：

* 它会追加，除非我们提供了消息 ID。
* 我们提供消息 ID 来覆盖消息，而不是追加到状态！

因此，要覆盖消息，我们只需提供消息 ID，即 `to_fork.values["messages"].id`。

In [ ]:
fork_config = graph.update_state(
    to_fork.config,
    {"messages": [HumanMessage(content='Multiply 5 and 3', 
                               id=to_fork.values["messages"][0].id)]},
)

In [ ]:
fork_config

这将创建一个新的、已分叉的检查点。

但是，元数据（例如，下一步要去哪里）被保留了！

我们可以看到 Agent 的当前状态已通过我们的分叉进行了更新。

In [ ]:
all_states = [state for state in graph.get_state_history(thread) ]
all_states[0].values["messages"]

In [ ]:
graph.get_state({'configurable': {'thread_id': '1'}})

现在，当我们进行流式传输时，图知道此检查点从未被执行过。

因此，图会运行，而不是简单地重放。

In [ ]:
for event in graph.stream(None, fork_config, stream_mode="values"):
    event['messages'][-1].pretty_print()

现在，我们可以看到当前状态是 Agent 运行的终点。

In [ ]:
graph.get_state({'configurable': {'thread_id': '1'}})

### 通过 LangGraph API 进行时间旅行

**⚠️ 注意**

自录制这些视频以来，我们已更新了 Studio，现在可以在本地运行并通过浏览器访问。这是运行 Studio 的首选方式，而非视频中展示的桌面应用。它现在名为 _LangSmith Studio_（而非 _LangGraph Studio_）。详细设置说明请参见课程开头的"环境搭建"指南。你可以在[此处](https://docs.langchain.com/langsmith/studio)找到 Studio 的说明，在[此处](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)找到本地部署的具体细节。  
要启动本地开发服务器，请在本模块的 `/studio` 目录下的终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到上面显示的 **Studio UI** URL。

我们通过 SDK 连接到它，并展示 LangGraph API 如何[支持时间旅行](https://docs.langchain.com/langsmith/human-in-the-loop-time-travel)。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangGraph Studio is currently not supported on Google Colab")

In [ ]:
from langgraph_sdk import get_client
client = get_client(url="http://127.0.0.1:2024")

#### 重放

让我们运行我们的 Agent，流式传输每个节点调用后的图状态 `updates`。

In [ ]:
initial_input = {"messages": HumanMessage(content="Multiply 2 and 3")}
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id = "agent",
    input=initial_input,
    stream_mode="updates",
):
    if chunk.data:
        assisant_node = chunk.data.get('assistant', {}).get('messages', [])
        tool_node = chunk.data.get('tools', {}).get('messages', [])
        if assisant_node:
            print("-" * 20+"Assistant Node"+"-" * 20)
            print(assisant_node[-1])
        elif tool_node:
            print("-" * 20+"Tools Node"+"-" * 20)
            print(tool_node[-1])

现在，让我们查看从指定检查点进行**重放**。

我们只需要传入 `checkpoint_id`。

In [ ]:
states = await client.threads.get_history(thread['thread_id'])
to_replay = states[-2]
to_replay

让我们使用 `stream_mode="values"` 进行流式传输，以查看重放时每个节点的完整状态。

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=None,
    stream_mode="values",
    checkpoint_id=to_replay['checkpoint_id']
):      
    print(f"Receiving new event of type: {chunk.event}...")
    print(chunk.data)
    print("\n\n")

我们还可以将其视为仅流式传输我们重放的节点对状态所做的 `updates`。

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=None,
    stream_mode="updates",
    checkpoint_id=to_replay['checkpoint_id']
):
    if chunk.data:
        assisant_node = chunk.data.get('assistant', {}).get('messages', [])
        tool_node = chunk.data.get('tools', {}).get('messages', [])
        if assisant_node:
            print("-" * 20+"Assistant Node"+"-" * 20)
            print(assisant_node[-1])
        elif tool_node:
            print("-" * 20+"Tools Node"+"-" * 20)
            print(tool_node[-1])

#### 分叉

现在，让我们看看分叉。

让我们获取与上面相同的步骤，即人工输入。

让我们为 Agent 创建一个新的线程。

In [ ]:
initial_input = {"messages": HumanMessage(content="Multiply 2 and 3")}
thread = await client.threads.create()
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=initial_input,
    stream_mode="updates",
):
    if chunk.data:
        assisant_node = chunk.data.get('assistant', {}).get('messages', [])
        tool_node = chunk.data.get('tools', {}).get('messages', [])
        if assisant_node:
            print("-" * 20+"Assistant Node"+"-" * 20)
            print(assisant_node[-1])
        elif tool_node:
            print("-" * 20+"Tools Node"+"-" * 20)
            print(tool_node[-1])

In [ ]:
states = await client.threads.get_history(thread['thread_id'])
to_fork = states[-2]
to_fork['values']

In [ ]:
to_fork['values']['messages'][0]['id']

In [ ]:
to_fork['next']

In [ ]:
to_fork['checkpoint_id']

让我们编辑状态。

请记住我们的 `messages` reducer 是如何工作的：

* 它会追加，除非我们提供了消息 ID。
* 我们提供消息 ID 来覆盖消息，而不是追加到状态！

In [ ]:
forked_input = {"messages": HumanMessage(content="Multiply 3 and 3",
                                         id=to_fork['values']['messages'][0]['id'])}

forked_config = await client.threads.update_state(
    thread["thread_id"],
    forked_input,
    checkpoint_id=to_fork['checkpoint_id']
)

In [ ]:
forked_config

In [ ]:
states = await client.threads.get_history(thread['thread_id'])
states[0]

要重新运行，我们传入 `checkpoint_id`。

In [ ]:
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=None,
    stream_mode="updates",
    checkpoint_id=forked_config['checkpoint_id']
):
    if chunk.data:
        assisant_node = chunk.data.get('assistant', {}).get('messages', [])
        tool_node = chunk.data.get('tools', {}).get('messages', [])
        if assisant_node:
            print("-" * 20+"Assistant Node"+"-" * 20)
            print(assisant_node[-1])
        elif tool_node:
            print("-" * 20+"Tools Node"+"-" * 20)
            print(tool_node[-1])

### LangGraph Studio

让我们在 Studio UI 中使用我们的 `agent`（使用 `module-1/studio/agent.py`，配置在 `module-1/studio/langgraph.json` 中）来查看分叉。